# Onde novos MW eólicos/solares evitam mais térmica fóssil — análise 2025

Notebook do projeto (base `projeto_energia_csv_2025.zip`). Roda de cima para baixo; cada etapa mostra uma tabela ou figura para conferência.

**Método:** (1) fator de capacidade (CF) horário típico por célula ERA5 de 0,5° e calibração contra as usinas existentes; (2) valor de cada MWh = fator marginal do MCTI × (1 − taxa de corte do subsistema), por mês e hora; (3) métrica principal = tCO₂ evitadas por MW instalado por ano; (4) clustering para regionalizar e ordenar regiões; (5) sensibilidades; (6) deslocamento marginal empírico (opcional).

**Limitações:** um ano (2025, seca no 2º semestre); fator marginal único para o SIN; corte médio ≠ corte marginal; sem transmissão, uso do solo ou custos — o ranking é de *potencial de abatimento*, não de viabilidade.

## 0. Preparação (Drive, zip, pasta de resultados)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)   # relê o Drive (pega uploads recentes)
import os, glob, zipfile, shutil

TAM_OK = 88051547            # tamanho exato do zip correto (versão de 2026-10-09)
PASTA_GRUPO = 'ENG4040_33A_GRUPOB'
ZIP_MANUAL = None            # se a busca falhar, cole aqui o caminho do zip, ex.: '/content/drive/Shareddrives/ENG4040_33A_GRUPOB/sprint 2/projeto_energia_csv_2025.zip'

def acha_zip():
    if ZIP_MANUAL:
        return ZIP_MANUAL
    raizes = ['/content/drive/Shareddrives', '/content/drive/MyDrive', '/content/drive/.shortcut-targets-by-id']
    vistos = []
    # 1º: dentro da pasta do grupo (drive compartilhado, atalho ou Meu Drive); 2º: qualquer lugar
    for padrao in (f'**/{PASTA_GRUPO}/**/*.zip', '**/projeto_energia_csv_2025*.zip'):
        for r in raizes:
            if not os.path.isdir(r):
                continue
            for p in glob.glob(f'{r}/{padrao}', recursive=True):
                t = os.path.getsize(p); vistos.append((t, p))
                if t == TAM_OK:
                    return p
    print('zips encontrados (nenhum com o tamanho certo):'); [print(f'  {t:>12,d}  {p}') for t, p in vistos]
    raise FileNotFoundError(
        f'Não achei o zip de {TAM_OK} bytes. Se a pasta {PASTA_GRUPO} foi "compartilhada comigo", '
        'clique nela com o botão direito no Drive → Organizar → Adicionar atalho ao Meu Drive, e rode de novo. '
        'Ou preencha ZIP_MANUAL com o caminho do zip.')

Z = acha_zip()
assert os.path.getsize(Z) == TAM_OK, f'{Z} não é a versão correta ({os.path.getsize(Z)} bytes)'
OUT = '/content/drive/MyDrive/resultados_analise_2025'   # no Drive de quem roda (a pasta do grupo pode ser só leitura)
DATA = '/content/extracted_data'
shutil.rmtree(DATA, ignore_errors=True)
zipfile.ZipFile(Z).extractall(DATA)
os.makedirs(OUT, exist_ok=True)
print('usando:', Z, '|', len(os.listdir(DATA)), 'arquivos extraídos | resultados em', OUT)

In [ ]:
import math
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 220)

cel = pd.read_csv(f'{DATA}/dim_celula.csv')
cel = cel[cel['terrestre'] == 1].reset_index(drop=True)
clima = pd.read_csv(f'{DATA}/clima_perfil_mes_hora.csv')
clima = clima[clima['id_celula'].isin(cel['id_celula'])].reset_index(drop=True)
ps = pd.read_csv(f'{DATA}/painel_subsistema_h.csv', parse_dates=['ts_local'])
pu = pd.read_csv(f'{DATA}/painel_uf_h.csv', parse_dates=['ts_local'])
usinas = pd.read_csv(f'{DATA}/dim_usina.csv')

DIAS = {m: pd.Period(f'2025-{m:02d}').days_in_month for m in range(1, 13)}
print('células terrestres:', len(cel), '| linhas de clima:', len(clima), '(esperado 2842 × 12 × 24 = 818 496)')
print('painel subsistema:', ps.shape, ps['ts_local'].min(), '→', ps['ts_local'].max())
print('painel UF:', pu.shape, '| usinas:', usinas.shape)

## 1. Fator de capacidade por célula × mês × hora

**Eólica:** curva genérica de turbina a 100 m — 0 abaixo de 3 m/s, cúbica entre 3 e 12, 1 entre 12 e 25, 0 acima de 25 —, integrada sobre uma distribuição de Weibull ajustada à média e ao desvio do vento em cada célula × mês × hora (`k = (dp/média)^−1,086`, `c = média/Γ(1+1/k)`; a curva é não linear, por isso não se aplica à média), com perdas de 12 %.
**Solar:** `CF = GHI/1000 × 0,80 × [1 − 0,004·(T_célula − 25)]`, `T_célula = T2m + 0,03·GHI`.

In [ ]:
V = np.arange(0, 35.0001, 0.1); DV = 0.1
PV = np.select([V < 3, V < 12, V < 25], [0.0, (V**3 - 3**3) / (12**3 - 3**3), 1.0], 0.0)
mu = np.maximum(clima['ws100_ms_media'].to_numpy(), 1e-3)
sd = np.maximum(clima['ws100_ms_dp'].to_numpy(), 1e-3)
K_W = np.clip((sd / mu) ** -1.086, 1.0, 12.0)
C_W = mu / np.vectorize(math.gamma)(1 + 1 / K_W)

def cf_eolico(idx, alfa=1.0):
    """CF esperado (com perdas de 12 %) para as linhas idx de `clima`, com o vento escalado por alfa."""
    k, c = K_W[idx], C_W[idx] * alfa
    out = np.empty(len(k))
    for i in range(0, len(k), 50_000):
        kk, cc = k[i:i + 50_000, None], c[i:i + 50_000, None]
        x = V[None, :] / cc
        f = (kk / cc) * x ** (kk - 1) * np.exp(-x ** kk)          # densidade de Weibull
        out[i:i + 50_000] = (f * PV).sum(axis=1) * DV
    return np.clip(0.88 * out, 0, 1)

TODAS = np.arange(len(clima))
clima['cf_eolica_bruto'] = cf_eolico(TODAS)
ghi, t2m = clima['ghi_wm2_media'], clima['t2m_c_media']
clima['cf_solar_bruto'] = (ghi / 1000 * 0.80 * (1 - 0.004 * (t2m + 0.03 * ghi - 25))).clip(0, 1)
clima['dias'] = clima['mes'].map(DIAS)

def cf_anual(col):
    g = clima.assign(w=clima[col] * clima['dias']).groupby('id_celula')[['w', 'dias']].sum()
    return g['w'] / g['dias']

anual = cel.set_index('id_celula')[['lat', 'lon', 'uf', 'subsistema', 'municipio']].copy()
anual['cf_eolica_bruto'] = cf_anual('cf_eolica_bruto'); anual['cf_solar_bruto'] = cf_anual('cf_solar_bruto')
anual[['cf_eolica_bruto', 'cf_solar_bruto']].describe().round(3)

### 1b. Calibração contra as usinas existentes
**CF observado por UF em 2025** (eólica/solar com ≥ 200 MW): (geração verificada + corte) / capacidade instalada, calculado da tabela oficial de fator de capacidade do ONS por usina (`fator_capacidade_renovavel_h`) e do constrained-off (`curtailment_h`) — tabelas do pacote *completo*; os valores ficam embutidos abaixo. Soma-se o corte porque o recurso não depende dele. (Não use `cap_*_mw` do `painel_uf_h` para isso: a capacidade ali vem do cadastro do ONS e não inclui as pequenas usinas, o que infla o CF, sobretudo na solar.)

**CF previsto por UF** = média do CF das células onde há usinas, ponderada pela potência.
- **Eólica:** o ERA5 a 0,5° subestima o vento em serras e as turbinas brasileiras atingem a potência nominal com menos vento; em vez de multiplicar o CF (o que distorce a curva), calibra-se um fator de escala do vento **α** tal que o CF previsto ponderado pela capacidade iguale o observado.
- **Solar:** fator multiplicativo único (inversor/DC-AC e rastreadores elevam o CF em relação à irradiância).

In [ ]:
# CF observado 2025 (geração + corte) / capacidade — ONS fator_capacidade_renovavel_h + curtailment_h (calculado na base)
CF_OBS = pd.DataFrame([
    ('eolica', 'BA', 11399, 0.515), ('eolica', 'CE', 2445, 0.428), ('eolica', 'MA', 426, 0.571), ('eolica', 'PB', 951, 0.524),
    ('eolica', 'PE', 820, 0.503), ('eolica', 'PI', 4646, 0.511), ('eolica', 'RN', 10292, 0.492), ('eolica', 'RS', 2027, 0.347),
    ('eolica', 'SC', 222, 0.322),
    ('solar', 'BA', 2337, 0.293), ('solar', 'CE', 1598, 0.271), ('solar', 'MG', 7209, 0.288), ('solar', 'PB', 584, 0.260),
    ('solar', 'PE', 974, 0.303), ('solar', 'PI', 2118, 0.264), ('solar', 'RN', 1874, 0.287), ('solar', 'SP', 1162, 0.240),
], columns=['fonte', 'uf', 'cap_mw', 'cf_observado'])

u = usinas.dropna(subset=['lat', 'lon', 'potencia_mw'])
u = u[(u['potencia_mw'] > 0) & u['grupo'].isin(['eolica', 'solar'])].copy()
u['lat_c'] = np.round(u['lat'] * 2) / 2; u['lon_c'] = np.round(u['lon'] * 2) / 2
u = u.merge(anual.reset_index()[['id_celula', 'lat', 'lon']], left_on=['lat_c', 'lon_c'], right_on=['lat', 'lon'], suffixes=('', '_cel'))
print(f"usinas eólicas/solares casadas a uma célula terrestre: {len(u)}")

def previsto(f, cf_cel):
    d = u[u['grupo'] == f].assign(cf=lambda x: x['id_celula'].map(cf_cel))
    return d.groupby('uf').apply(lambda x: np.average(x['cf'], weights=x['potencia_mw'])).rename('cf_previsto')

def media_ponderada(f, prev):
    o = CF_OBS[CF_OBS['fonte'] == f].set_index('uf').join(prev, how='inner')
    return np.average(o['cf_previsto'], weights=o['cap_mw']), np.average(o['cf_observado'], weights=o['cap_mw'])

# eólica: α por bissecção (só as linhas de clima das células com parque, para ser rápido)
cel_parque = u.loc[u['grupo'] == 'eolica', 'id_celula'].unique()
idx_parque = np.flatnonzero(clima['id_celula'].isin(cel_parque).to_numpy())
def prev_eol(alfa):
    sub = clima.iloc[idx_parque][['id_celula', 'dias']].assign(cf=cf_eolico(idx_parque, alfa))
    g = sub.assign(w=sub['cf'] * sub['dias']).groupby('id_celula')[['w', 'dias']].sum()
    return previsto('eolica', g['w'] / g['dias'])
lo, hi = 0.8, 2.5
for _ in range(25):
    mid = (lo + hi) / 2
    p, o = media_ponderada('eolica', prev_eol(mid))
    lo, hi = (mid, hi) if p < o else (lo, mid)
ALFA = (lo + hi) / 2
clima['cf_eolica'] = cf_eolico(TODAS, ALFA)

# solar: fator multiplicativo
p, o = media_ponderada('solar', previsto('solar', anual['cf_solar_bruto']))
FATOR_SOL = o / p
clima['cf_solar'] = (clima['cf_solar_bruto'] * FATOR_SOL).clip(0, 1)

anual['cf_eolica'] = cf_anual('cf_eolica'); anual['cf_solar'] = cf_anual('cf_solar')
print(f'Escala do vento α = {ALFA:.3f} | fator solar = {FATOR_SOL:.3f}')

cal = pd.concat([
    CF_OBS[CF_OBS['fonte'] == f].set_index('uf')
      .join(previsto(f, anual[f'cf_{f}_bruto']).rename('cf_previsto_bruto'))
      .join(previsto(f, anual[f'cf_{f}']).rename('cf_previsto_calibrado'))
    for f in ('eolica', 'solar')]).reset_index()
cal['erro_calibrado_pp'] = 100 * (cal['cf_previsto_calibrado'] - cal['cf_observado'])
cal.round(3)

## 2. Valor de um MWh por subsistema × mês × hora
`om` = fator marginal oficial do MCTI (tCO₂/MWh, igual nos 4 subsistemas). `r` = taxa de corte (corte/referência) da fonte no subsistema. **Valor efetivo** `E = om × (1 − r)`: um MW novo num subsistema e hora com corte tende a ser cortado pelo menos nessa taxa.

In [ ]:
ps['mes'] = ps['ts_local'].dt.month
ps['hora_local'] = ps['ts_local'].dt.hour
ger_cols = [c for c in ps.columns if c.startswith('ger_') and c.endswith('_mwh') and 'sem_fator' not in c and 'fora_sin' not in c]
ps['ger_total_mwh'] = ps[ger_cols].sum(axis=1)

val = ps.groupby(['subsistema', 'mes', 'hora_local']).agg(
    om=('fator_om_mcti_tco2_mwh', 'mean'),
    corte_e=('curtail_corte_mwh_eolica', 'sum'), ref_e=('ref_eolica_mwh', 'sum'),
    corte_s=('curtail_corte_mwh_solar', 'sum'), ref_s=('ref_solar_mwh', 'sum'),
    fossil_om=('term_fossil_ordemmerito_mwh', 'sum'), fossil_tot=('term_fossil_total_mwh', 'sum'),
    emis=('emissoes_estimadas_tco2', 'sum'), ger=('ger_total_mwh', 'sum'), pld=('pld_rs_mwh', 'mean')).reset_index()
val['r_eolica'] = np.where(val['ref_e'] > 0, val['corte_e'] / val['ref_e'], 0.0).clip(0, 1)
val['r_solar'] = np.where(val['ref_s'] > 0, val['corte_s'] / val['ref_s'], 0.0).clip(0, 1)
# subsistema sem NENHUM dado de corte da fonte no ano → r = 0 é ausência de dado, não corte zero
SEM_DADO = {f: [s for s, d in val.groupby('subsistema') if d[ref].sum() == 0] for f, ref in (('eolica', 'ref_e'), ('solar', 'ref_s'))}
print('subsistemas sem dado de corte:', SEM_DADO)
ne = val[val['subsistema'] == 'NE'].set_index(['mes', 'hora_local'])
for f in ('eolica', 'solar'):
    r_ne = val.set_index(['mes', 'hora_local']).index.map(ne[f'r_{f}'])
    val[f'r_{f}_ne'] = np.where(val['subsistema'].isin(SEM_DADO[f]), r_ne, val[f'r_{f}'])
val['intens_media'] = val['emis'] / val['ger']
val['frac_desp'] = np.where(val['fossil_tot'] > 0, val['fossil_om'] / val['fossil_tot'], 0.0).clip(0, 1)
for f in ('eolica', 'solar'):
    val[f'E_{f}'] = val['om'] * (1 - val[f'r_{f}'])                          # principal
    val[f'E_{f}_sem_corte'] = val['om']                                       # sensibilidade 1
    val[f'E_{f}_intens_media'] = val['intens_media'] * (1 - val[f'r_{f}'])   # sensibilidade 2
    val[f'E_{f}_desp'] = val['om'] * val['frac_desp'] * (1 - val[f'r_{f}'])  # sensibilidade 3
    val[f'E_{f}_pld'] = val['pld'] * (1 - val[f'r_{f}'])                     # sensibilidade 4 (R$/MWh)
    val[f'E_{f}_corte_ne'] = val['om'] * (1 - val[f'r_{f}_ne'])              # sensibilidade 5: sem dado → corte do NE

print('Taxa de corte média por subsistema (média simples das células mês×hora):')
val.groupby('subsistema')[['om', 'r_eolica', 'r_solar', 'frac_desp']].mean().round(3)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
om = val[val['subsistema'] == 'SE'].pivot(index='mes', columns='hora_local', values='om')
for a, (titulo, m) in zip(ax, [('OM do MCTI (tCO₂/MWh) — SIN', om),
                               ('Corte eólico NE (fração)', val[val['subsistema'] == 'NE'].pivot(index='mes', columns='hora_local', values='r_eolica')),
                               ('Corte solar NE (fração)', val[val['subsistema'] == 'NE'].pivot(index='mes', columns='hora_local', values='r_solar'))]):
    im = a.imshow(m, aspect='auto', cmap='viridis', origin='lower', extent=[-0.5, 23.5, 0.5, 12.5])
    a.set_title(titulo); a.set_xlabel('hora local'); a.set_ylabel('mês'); fig.colorbar(im, ax=a)
plt.tight_layout(); plt.savefig(f'{OUT}/fig_valor_heatmaps.png', dpi=130); plt.show()

fig, ax = plt.subplots(2, 4, figsize=(18, 6), sharex=True, sharey=True)
for j, s in enumerate(['N', 'NE', 'S', 'SE']):
    for i, f in enumerate(['eolica', 'solar']):
        m = val[val['subsistema'] == s].pivot(index='mes', columns='hora_local', values=f'r_{f}')
        im = ax[i, j].imshow(m, aspect='auto', cmap='magma', vmin=0, vmax=0.8, origin='lower', extent=[-0.5, 23.5, 0.5, 12.5])
        ax[i, j].set_title(f'corte {f} — {s}')
fig.colorbar(im, ax=ax, shrink=0.8); plt.savefig(f'{OUT}/fig_corte_subsistemas.png', dpi=130); plt.show()

## 3. Métricas por célula
- `energia_mwh_por_mw` = Σ dias × CF (MWh por MW instalado no ano)
- `co2_evitado_t_por_mw` = Σ dias × CF × E  ← **métrica principal**
- `intensidade_evitada` = co2/energia (qualidade do horário, independente da quantidade de recurso)
- `perda_por_corte` = fração da energia gerada em horas de corte

In [ ]:
base = (clima[['id_celula', 'mes', 'hora_local', 'dias', 'cf_eolica', 'cf_solar']]
        .merge(cel[['id_celula', 'subsistema']], on='id_celula')
        .merge(val, on=['subsistema', 'mes', 'hora_local'], how='left'))
assert base['om'].notna().all(), 'faltou valor para alguma combinação subsistema × mês × hora'

def metricas(f, sufixo=''):
    w = base['dias'] * base[f'cf_{f}']
    g = pd.DataFrame({'id_celula': base['id_celula'], 'w': w, 'wE': w * base[f'E_{f}{sufixo}'],
                      'wr': w * (1 - base[f'r_{f}'])}).groupby('id_celula').sum()
    return g

scores = []
for f in ('eolica', 'solar'):
    g = metricas(f)
    s = pd.DataFrame({'fonte': f, 'cf_anual': g['w'] / 8760, 'energia_mwh_por_mw': g['w'],
                      'co2_evitado_t_por_mw': g['wE'], 'intensidade_evitada': g['wE'] / g['w'],
                      'perda_por_corte': 1 - g['wr'] / g['w']})
    for suf in ('_sem_corte', '_intens_media', '_desp', '_pld', '_corte_ne'):
        s[f'co2{suf}' if suf != '_pld' else 'receita_rs_por_mw'] = metricas(f, suf)['wE']
    scores.append(s)
scores = pd.concat(scores).reset_index().merge(cel[['id_celula', 'lat', 'lon', 'uf', 'municipio', 'subsistema']], on='id_celula')
scores['sem_dado_corte'] = [s in SEM_DADO[f] for f, s in zip(scores['fonte'], scores['subsistema'])]
scores.to_csv(f'{OUT}/scores_celulas.csv', index=False)
print('gravado scores_celulas.csv:', scores.shape)
scores.groupby('fonte')[['cf_anual', 'energia_mwh_por_mw', 'co2_evitado_t_por_mw', 'intensidade_evitada', 'perda_por_corte']].describe(percentiles=[.1, .5, .9]).T.round(3)

In [ ]:
for f in ('eolica', 'solar'):
    print(f'\nTop-20 células — {f} (tCO₂ evitadas por MW por ano)')
    display(scores[scores['fonte'] == f].nlargest(20, 'co2_evitado_t_por_mw')
            [['municipio', 'uf', 'subsistema', 'lat', 'lon', 'cf_anual', 'co2_evitado_t_por_mw', 'intensidade_evitada', 'perda_por_corte', 'sem_dado_corte']].round(3))

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(14, 13))
for i, f in enumerate(['eolica', 'solar']):
    d = scores[scores['fonte'] == f]
    for j, (col, tit) in enumerate([('cf_anual', 'CF anual'), ('co2_evitado_t_por_mw', 'tCO₂ evitadas / MW / ano')]):
        sc = ax[i, j].scatter(d['lon'], d['lat'], c=d[col], s=9, marker='s', cmap='YlOrRd')
        ax[i, j].set_title(f'{f} — {tit}'); ax[i, j].set_aspect('equal'); fig.colorbar(sc, ax=ax[i, j], shrink=0.7)
plt.tight_layout(); plt.savefig(f'{OUT}/fig_mapas_scores.png', dpi=130); plt.show()

## 4. Clustering (regionalização)
O cluster agrupa células com **regime** parecido (forma do perfil diário e sazonal, nível de recurso, horário e corte); depois os clusters são **ordenados** pela métrica principal — é essa ordenação que responde "onde". Células com recurso irrisório (CF anual abaixo do limiar) ficam fora.

Features: perfil de 24 h e de 12 meses divididos pelo CF anual (forma) → padronizados → PCA (90 % da variância); mais 3 escalares padronizados (CF anual, intensidade evitada, perda por corte), com peso que dá aos dois blocos a mesma variância total.

In [ ]:
CF_MIN = {'eolica': 0.15, 'solar': 0.12}   # limiar de recurso (CF anual calibrado)

def features(f):
    ok = scores[(scores['fonte'] == f) & (scores['cf_anual'] >= CF_MIN[f])].set_index('id_celula')
    d = clima[clima['id_celula'].isin(ok.index)]
    p24 = (d.assign(w=d['dias'] * d[f'cf_{f}']).groupby(['id_celula', 'hora_local'])['w'].sum()
           / d.groupby(['id_celula', 'hora_local'])['dias'].sum()).unstack()
    p12 = d.groupby(['id_celula', 'mes'])[f'cf_{f}'].mean().unstack()
    p24 = p24.div(ok['cf_anual'], axis=0).add_prefix('h'); p12 = p12.div(ok['cf_anual'], axis=0).add_prefix('m')
    esc = ok[['cf_anual', 'intensidade_evitada', 'perda_por_corte']]
    return ok, p24.join(p12).loc[ok.index], esc

for f in ('eolica', 'solar'):
    ok, forma, esc = features(f)
    print(f'{f}: {len(ok)} células com CF ≥ {CF_MIN[f]} (de {int((scores.fonte == f).sum())}); features de forma {forma.shape}')

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.neighbors import kneighbors_graph

K_FORCADO = {'eolica': None, 'solar': None}   # troque por um inteiro para fixar k
res = {}
for f in ('eolica', 'solar'):
    ok, forma, esc = features(f)
    zf = PCA(n_components=0.90, random_state=0).fit_transform(StandardScaler().fit_transform(forma))
    ze = StandardScaler().fit_transform(esc) * np.sqrt(zf.shape[1] / esc.shape[1])
    X = np.hstack([zf, ze])
    sel = []
    for k in range(3, 11):
        km = KMeans(n_clusters=k, n_init=20, random_state=0).fit(X)
        sel.append(dict(k=k, inercia=km.inertia_, silhueta=silhouette_score(X, km.labels_, sample_size=min(3000, len(X)), random_state=0)))
    sel = pd.DataFrame(sel)
    k = K_FORCADO[f] or int(sel[sel['k'].between(4, 8)].sort_values('silhueta', ascending=False)['k'].iloc[0])
    km = KMeans(n_clusters=k, n_init=20, random_state=0).fit(X)
    conn = kneighbors_graph(ok[['lat', 'lon']].to_numpy(), 8, include_self=False)
    ward = AgglomerativeClustering(n_clusters=k, linkage='ward', connectivity=conn).fit(X)
    res[f] = dict(ok=ok.assign(cluster_km=km.labels_, cluster_ward=ward.labels_), sel=sel, k=k, X=X)
    print(f'{f}: k = {k} (componentes PCA da forma: {zf.shape[1]}); concordância KMeans × Ward (ARI) = {adjusted_rand_score(km.labels_, ward.labels_):.2f}')

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for a, f in zip(ax, ('eolica', 'solar')):
    s = res[f]['sel']; a.plot(s['k'], s['silhueta'], 'o-'); a2 = a.twinx(); a2.plot(s['k'], s['inercia'], 's--', color='gray')
    a.axvline(res[f]['k'], color='red', ls=':'); a.set_title(f'{f}: silhueta (azul) e inércia (cinza)'); a.set_xlabel('k')
plt.tight_layout(); plt.savefig(f'{OUT}/fig_escolha_k.png', dpi=130); plt.show()

In [ ]:
def resumo_clusters(f, col='cluster_km'):
    d = res[f]['ok']
    r = d.groupby(col).agg(celulas=('cf_anual', 'size'),
                           ufs=('uf', lambda s: ', '.join(s.value_counts().index[:4])),
                           cf_anual=('cf_anual', 'median'),
                           co2_mediana=('co2_evitado_t_por_mw', 'median'),
                           co2_p10=('co2_evitado_t_por_mw', lambda s: s.quantile(.1)),
                           co2_p90=('co2_evitado_t_por_mw', lambda s: s.quantile(.9)),
                           intensidade=('intensidade_evitada', 'median'),
                           perda_corte=('perda_por_corte', 'median'))
    r = r.sort_values('co2_mediana', ascending=False)
    r.insert(0, 'rank', range(1, len(r) + 1))
    res[f]['ok']['rank_cluster'] = res[f]['ok'][col].map(r['rank'])
    return r

for f in ('eolica', 'solar'):
    r = resumo_clusters(f)
    r.to_csv(f'{OUT}/resumo_clusters_{f}.csv')
    print(f'\nClusters — {f} (rank 1 = mais tCO₂ evitada por MW)'); display(r.round(3))

In [ ]:
ex = usinas.dropna(subset=['lat', 'lon', 'potencia_mw'])
fig, ax = plt.subplots(1, 2, figsize=(16, 8))
for a, f in zip(ax, ('eolica', 'solar')):
    d = res[f]['ok']
    sc = a.scatter(d['lon'], d['lat'], c=d['rank_cluster'], cmap='RdYlGn_r', s=12, marker='s')
    e = ex[ex['grupo'] == f]
    a.scatter(e['lon'], e['lat'], s=e['potencia_mw'] / 15, facecolors='none', edgecolors='k', lw=0.6, label='usinas existentes')
    a.set_aspect('equal'); a.set_title(f'{f}: clusters ordenados (1 = melhor)'); a.legend(loc='lower left')
    fig.colorbar(sc, ax=a, shrink=0.6, label='rank do cluster')
plt.tight_layout(); plt.savefig(f'{OUT}/fig_mapa_clusters.png', dpi=130); plt.show()

fig, ax = plt.subplots(2, 2, figsize=(15, 8))
for i, f in enumerate(('eolica', 'solar')):
    d = res[f]['ok'][['rank_cluster']].join(clima.set_index('id_celula'), how='inner')
    for r, g in d.groupby('rank_cluster'):
        ax[i, 0].plot(g.groupby('hora_local')[f'cf_{f}'].mean(), label=f'cluster {r}')
        ax[i, 1].plot(g.groupby('mes')[f'cf_{f}'].mean(), label=f'cluster {r}')
    ax[i, 0].set_title(f'{f}: perfil diário médio (CF)'); ax[i, 1].set_title(f'{f}: perfil sazonal médio (CF)'); ax[i, 0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT}/fig_perfis_clusters.png', dpi=130); plt.show()

for f in ('eolica', 'solar'):
    d = res[f]['ok']
    top = d.nlargest(20, 'co2_evitado_t_por_mw')[['municipio', 'uf', 'subsistema', 'rank_cluster', 'cf_anual', 'co2_evitado_t_por_mw', 'perda_por_corte']]
    top.to_csv(f'{OUT}/top20_{f}.csv'); print(f'\nTop-20 {f} com cluster:'); display(top.round(3))
    print(f'Células de alto CF mas fora do top-20 % em CO₂ (muito recurso, pouco valor):')
    alto_cf = d[d['cf_anual'] >= d['cf_anual'].quantile(.8)]
    print(alto_cf[alto_cf['co2_evitado_t_por_mw'] < d['co2_evitado_t_por_mw'].quantile(.8)].groupby('uf').size().sort_values(ascending=False).head(8).to_dict())

## 5. Sensibilidade — o ranking muda com a definição de valor?
1. sem desconto de corte; 2. intensidade média do subsistema em vez do OM; 3. só a fração fóssil despachada por ordem de mérito; 4. nos subsistemas **sem dado de corte** da fonte (solar no N e no S), aplica a taxa de corte do NE na mesma hora/mês (cenário conservador); 5. valor econômico (PLD × (1 − corte), R$/MW/ano). Medidas: correlação de Spearman com o ranking principal e sobreposição do top-10 %.

In [ ]:
alts = {'sem corte': 'co2_sem_corte', 'intensidade média': 'co2_intens_media', 'fóssil despachável': 'co2_desp',
        'corte do NE onde não há dado': 'co2_corte_ne', 'receita PLD': 'receita_rs_por_mw'}
linhas = []
for f in ('eolica', 'solar'):
    d = scores[scores['fonte'] == f]
    top = set(d.nlargest(int(len(d) * .1), 'co2_evitado_t_por_mw')['id_celula'])
    for nome, col in alts.items():
        topa = set(d.nlargest(int(len(d) * .1), col)['id_celula'])
        linhas.append(dict(fonte=f, alternativa=nome, spearman=d['co2_evitado_t_por_mw'].rank().corr(d[col].rank()),
                           sobreposicao_top10=len(top & topa) / len(top)))
sens = pd.DataFrame(linhas); sens.to_csv(f'{OUT}/sensibilidade.csv', index=False)
sens.round(3)

## 6. (Opcional) Deslocamento marginal empírico
Regressão horária das variações da geração fóssil do SIN contra as variações de eólica e solar de cada subsistema, controlando a variação da carga e com efeitos fixos de mês × hora (por demeaning). O coeficiente indica quantos MWh fósseis saem por MWh renovável a mais — indicativo (a hidrelétrica absorve boa parte da variação).

In [ ]:
w = ps.pivot_table(index='ts_local', columns='subsistema', values=['ger_eolica_mwh', 'ger_solar_mwh', 'ger_fossil_mwh', 'carga_global_mwh'], aggfunc='sum')
y = w['ger_fossil_mwh'].sum(axis=1).diff()
X = pd.concat({f'{v}_{s}': w[f'ger_{v}_mwh'][s].diff() for v in ('eolica', 'solar') for s in ('N', 'NE', 'S', 'SE')}, axis=1)
X['carga_SIN'] = w['carga_global_mwh'].sum(axis=1).diff()
D = pd.concat([y.rename('y'), X], axis=1).dropna()
grp = [D.index.month, D.index.hour]
D = D - D.groupby(grp).transform('mean')            # efeitos fixos mês × hora
Xm, ym = D.drop(columns='y').to_numpy(), D['y'].to_numpy()
beta, *_ = np.linalg.lstsq(Xm, ym, rcond=None)
resid = ym - Xm @ beta
se = np.sqrt(np.diag(np.linalg.inv(Xm.T @ Xm)) * resid.var(ddof=Xm.shape[1]))
reg = pd.DataFrame({'coef_mwh_fossil_por_mwh': beta, 'erro_padrao': se}, index=D.drop(columns='y').columns)
reg['t'] = reg['coef_mwh_fossil_por_mwh'] / reg['erro_padrao']
reg.to_csv(f'{OUT}/regressao_deslocamento.csv')
reg.round(4)

## 7. Síntese

In [ ]:
for f in ('eolica', 'solar'):
    r = resumo_clusters(f)
    b = r.iloc[0]
    print(f"{f.upper()}: melhor cluster = UFs {b['ufs']} — {int(b['celulas'])} células, CF mediano {b['cf_anual']:.2f}, "
          f"{b['co2_mediana']:.0f} tCO₂/MW/ano (p10–p90: {b['co2_p10']:.0f}–{b['co2_p90']:.0f}), perda por corte {b['perda_corte']:.0%}")
    s = sens[sens['fonte'] == f].set_index('alternativa')['sobreposicao_top10']
    print('   robustez (sobreposição do top-10 %):', {k: round(v, 2) for k, v in s.items()})
print(f'\nArquivos gravados em {OUT}:', sorted(os.listdir(OUT)))